# California 2035 — one scenario

Solves the PyPSA-USA California 2035 network for **one scenario** with the same constraints as
Snakemake, then writes tables and figures to `outputs/<SCENARIO>/`, where `SCENARIO` is named from the parameters. The logic lives in
`ca_helpers.py`. Kernel: `pypsa-usa/.venv/bin/python` (see README for setup).

## Parameters

In [ ]:
# Prepared network: "hybrid25" (25 nodes, the base, ~15 min; see spatial_resolution.ipynb) or "zones4" (4 ReEDS zones, 2-4 min)
NETWORK = "hybrid25"

PARAMS = {
    "export_price": 40,  # $/MWh paid for exports (≈ CAISO 2019 day-ahead average)
    "import_price": 60,  # $/MWh for unspecified imports
    "erm_margin": 0.15,  # energy reserve margin (fraction above load)
    # Renewable policy for California (on top of the AB 32 CO2 cap):
    #   None     = base case, no renewable target
    #   "annual" = over the year, eligible generation minus exports >= renewable_share x CA load
    #              (SB 100 style: gas can still run in some hours if other hours compensate)
    #   "hourly" = hour-by-hour matching ("24/7"): each hour, clean supply counts only up to that hour's load
    #              (no surplus carried to other hours); the matched total >= renewable_share x CA load
    #   "strict" = 100% every hour: fossil plants and unspecified imports are switched off
    "renewables": 'annual',
    "renewable_share": 1.0,   # for "annual" and "hourly" (1.0 = 100%, 0.95 = 95%)
    "nuclear_counts": True,  # True: nuclear (Diablo Canyon) counts as clean and stays available, as in SB 100
    "new_nuclear": False,  # False: no new nuclear plants (California moratorium), only existing units
}

# Results go to outputs/<name>/, named from the parameters: {buses}bus_{policy}_{year}_{non-default settings}_{TAG}
# e.g. 25bus_re95-hourly_2035_nuc. TAG is an optional extra label (e.g. "wave"); None for none.
TAG = None

FORCE = False  # True = re-solve even if outputs/<SCENARIO>/ already holds a result with these parameters

## Solve

In [ ]:
# reload ca_helpers.py when it changes, without restarting the kernel
%load_ext autoreload
%autoreload 2
import ca_helpers as ch

SCENARIO = ch.scenario_name(PARAMS, NETWORK, TAG)
print(SCENARIO)

n = ch.run_scenario(SCENARIO, PARAMS, force=FORCE, network=NETWORK)
ch.check_against_snakemake(n, PARAMS, network=NETWORK)  # only runs when PARAMS equal the Snakemake base config

## Tables

In [ ]:
ch.show_tables(n)

## Figures

In [ ]:
ch.show_figures(n)

## Save results

In [ ]:
ch.export(n, SCENARIO)

## Compare with other scenarios in `outputs/`

In [ ]:
ch.compare_scenarios()